# клиентская аналитика в Python

Этот ноутбук можно использовать как дополнительный практический блок, если после основной части лекции останется время.

Внутри мы:
- загрузим данные о клиентах и заказах;
- отфильтруем только оплаченные заказы;
- объединим таблицы;
- построим клиентские признаки;
- выполним простую сегментацию;
- сохраним результат в CSV.


## 1. Подключение библиотек

Для практики используем:
- `pathlib` для работы с путями;
- `pandas` для таблиц;
- `numpy` для простой прикладной логики сегментации.


In [2]:
from pathlib import Path

import pandas as pd
import numpy as np


## 2. Пути к данным

Сначала определим, где находятся входные файлы проекта.


In [3]:
BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = BASE_DIR / "data"

customers_path = DATA_DIR / "customers.csv"
orders_path = DATA_DIR / "orders.csv"

print("BASE_DIR =", BASE_DIR)
print("customers_path =", customers_path)
print("orders_path =", orders_path)
print("customers.csv существует:", customers_path.exists())
print("orders.csv существует:", orders_path.exists())


BASE_DIR = C:\TLTSU\2025-2026 учебный год\Дисциплины\Лекция Акрон\customer_analytics_pipeline
customers_path = C:\TLTSU\2025-2026 учебный год\Дисциплины\Лекция Акрон\customer_analytics_pipeline\data\customers.csv
orders_path = C:\TLTSU\2025-2026 учебный год\Дисциплины\Лекция Акрон\customer_analytics_pipeline\data\orders.csv
customers.csv существует: True
orders.csv существует: True


## 3. Загрузка исходных данных

Считаем CSV-файлы в DataFrame и посмотрим на первые строки.


In [4]:
customers_df = pd.read_csv(customers_path)
orders_df = pd.read_csv(orders_path)

print("Размер customers_df:", customers_df.shape)
print("Размер orders_df:", orders_df.shape)

customers_df.head()


Размер customers_df: (8, 4)
Размер orders_df: (15, 6)


,customer_id,name,city,segment
0,1,Анна,Москва,B2C
1,2,Иван,Казань,B2C
2,3,Ольга,Самара,B2B
3,4,Дмитрий,Москва,B2C
4,5,Мария,Санкт-Петербург,B2B


In [5]:
orders_df.head()


,order_id,customer_id,order_date,amount,category,status
0,101,1,2026-03-01,1500,Электроника,paid
1,102,2,2026-03-02,700,Книги,paid
2,103,1,2026-03-03,2300,Бытовая техника,paid
3,104,3,2026-03-04,1200,Книги,paid
4,105,4,2026-03-05,3200,Электроника,paid


## 4. Отбор только оплаченных заказов

Для аналитики часто полезно сразу убрать нерелевантные записи.
Оставим только заказы со статусом `paid`.


In [ ]:
paid_orders_df = orders_df[orders_df["status"] == "paid"].copy()

print("Всего заказов:", len(orders_df))
print("Оплаченных заказов:", len(paid_orders_df))

paid_orders_df.head()


## 5. Объединение заказов и клиентов

Теперь соединим данные о заказах и данные о клиентах по `customer_id`.


In [ ]:
merged_df = paid_orders_df.merge(customers_df, on="customer_id", how="left")

print("Размер объединённой таблицы:", merged_df.shape)
merged_df.head()


## 6. Построение клиентских признаков

Сформируем по каждому клиенту агрегированные показатели:
- `total_spent` — суммарные траты;
- `orders_count` — количество заказов;
- `avg_check` — средний чек;
- `last_order_date` — дата последнего заказа.


In [ ]:
customer_features = (
    merged_df.groupby(["customer_id", "name", "city", "segment"])
             .agg(
                 total_spent=("amount", "sum"),
                 orders_count=("order_id", "count"),
                 avg_check=("amount", "mean"),
                 last_order_date=("order_date", "max")
             )
             .reset_index()
)

customer_features["total_spent"] = customer_features["total_spent"].round(2)
customer_features["avg_check"] = customer_features["avg_check"].round(2)

customer_features


## 7. Простая сегментация клиентов

Добавим простую прикладную логику:
- если суммарные траты клиента больше 3000 — сегмент `VIP`;
- иначе — сегмент `Regular`.


In [ ]:
customer_features["value_segment"] = np.where(
    customer_features["total_spent"] > 3000,
    "VIP",
    "Regular"
)

customer_features


## 8. Сортировка клиентов по ценности

Теперь отсортируем клиентов по суммарным тратам.


In [ ]:
top_customers = customer_features.sort_values("total_spent", ascending=False)
top_customers


## 9. Отбор только VIP-клиентов

Выделим только клиентов сегмента `VIP`.


In [ ]:
vip_customers = customer_features[customer_features["value_segment"] == "VIP"]
vip_customers


## 10. Сохранение результата

Сохраним итоговую клиентскую витрину в CSV-файл.


In [ ]:
output_path = DATA_DIR / "customer_features_optional_practice.csv"
customer_features.to_csv(output_path, index=False)

print("Файл сохранён:", output_path)


## 11. Проверка сохранённого файла

Проверим, что сохранённый результат можно корректно считать обратно.


In [ ]:
saved_df = pd.read_csv(output_path)
saved_df.head()


## 12. Итог мини-практики

В этой мини-практике мы:
- загрузили исходные данные;
- отфильтровали только оплаченные заказы;
- объединили таблицы;
- построили клиентские признаки;
- выполнили простую сегментацию;
- сохранили результат в файл.

Именно такие небольшие прикладные пайплайны часто лежат в основе аналитических и AI/ML-задач.
